In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

from src.utils import *
from src.analysis import *
from src.model import FineTunedModel



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FineTunedModel(num_classes=10).to(device)
model.load_state_dict(torch.load('weights/finetune_weights.pth', map_location=device))
target_layer = model.feature_extractor[5]
D = torch.load('weights/hals_nnd_D.pth')


In [ ]:
subset_paths = load_subset(subset_root="data/subset")
analysis_result = run_complete_analysis(
    model=model,
    target_layer=target_layer,
    D=torch.load('weights/hals_nnd_D.pth'),
    subset_paths=subset_paths,
    k=5,
    lam=1e-2,
    batch_size=10
)

In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/church/1.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")

In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/church/2.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")

In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/tench/2.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")

In [ ]:
def visualize_atom_overlap_matrix(overlap_matrix: np.ndarray, class_names: List[str]):
    """Visualize the atom overlap matrix as a heatmap."""
    plt.figure(figsize=(5, 4))
    
    # Mask diagonal
    mask = np.eye(10, dtype=bool)
    overlap_masked = np.ma.masked_where(mask, overlap_matrix)
    
    plt.imshow(overlap_masked, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(label='Jaccard Similarity')
    
    # Set ticks
    plt.xticks(range(10), class_names, rotation=45, ha='right', fontsize=6)
    plt.yticks(range(10), class_names, fontsize=6)
    
    # Add values
    for i in range(10):
        for j in range(10):
            if i != j and overlap_matrix[i, j] > 0:
                plt.text(j, i, f'{overlap_matrix[i, j]:.2f}', 
                        ha='center', va='center', fontsize=4)
    
    plt.title('Top-5 Characteristic Atoms Overlap Between Classes', 
              fontsize=7, fontweight='bold')
    plt.xlabel('Class')
    plt.ylabel('Class')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/atom_overlap_matrix.png', dpi=150, bbox_inches='tight')
    print("\nSaved atom overlap visualization to: figures/atom_overlap_matrix.png")
    plt.show()




In [ ]:
def analyze_class_characteristic_atoms(results: List[dict], n_atoms: int, 
                                       top_k_atoms: int = 10, threshold: float = 0.1,
                                       top_channels: int = 10):
    """
    Analyze which atoms are characteristic for each class.
    
    Args:
        results: List of analysis results with 'label', 'path', and 'analysis' keys
        n_atoms: Total number of atoms in dictionary
        top_k_atoms: Number of top atoms to report per class
        threshold: Minimum average weight to consider an atom as characteristic
        top_channels: Number of top influential channels to average (default: 10)
        
    Returns:
        Dictionary with class-specific atom statistics
    """
    # Extract class names from paths in results
    # Path format: 'data/subset/class_name/image.jpeg'
    class_name_set = set()
    label_to_class_name = {}
    
    for r in results:
        path = r['path']
        label = r['label']
        # Extract class name from path (second to last component)
        class_name = path.split('/')[-2]
        class_name_set.add(class_name)
        label_to_class_name[label] = class_name
    
    # Sort class names by their label indices
    sorted_labels = sorted(label_to_class_name.keys())
    class_names = [label_to_class_name[label] for label in sorted_labels]
    n_classes = len(class_names)
    
    print(f"\nDetected {n_classes} classes: {class_names}")
    
    # Aggregate sparse codes by class
    class_codes = {i: [] for i in range(n_classes)}
    
    for r in results:
        label = r['label']
        sparse_codes = r['analysis']['sparse_codes']  # (n_channels, n_atoms)
        weights = r['analysis']['weights']  # (n_channels,)
        
        # Get top-K most influential channels
        top_channel_indices = np.argsort(weights)[-top_channels:]
        
        # Average across only top-K channels for this image
        top_codes = sparse_codes[top_channel_indices]  # (top_channels, n_atoms)
        avg_code = np.mean(np.abs(top_codes), axis=0)  # (n_atoms,)
        class_codes[label].append(avg_code)
    
    # Compute statistics per class
    class_stats = {}
    
    for class_idx in range(n_classes):
        if len(class_codes[class_idx]) == 0:
            continue
        
        codes = np.array(class_codes[class_idx])  # (n_images, n_atoms)
        
        # Compute mean and std for each atom across images in this class
        mean_activation = np.mean(codes, axis=0)  # (n_atoms,)
        std_activation = np.std(codes, axis=0)
        
        # Compute consistency: how often each atom is used (non-zero)
        usage_rate = np.mean(codes > 1e-6, axis=0)  # (n_atoms,)
        
        class_stats[class_idx] = {
            'mean_activation': mean_activation,
            'std_activation': std_activation,
            'usage_rate': usage_rate,
            'n_images': len(codes)
        }
    
    # Find characteristic atoms for each class
    print("\n" + "="*80)
    print("CLASS-CHARACTERISTIC ATOMS ANALYSIS")
    print("="*80)
    
    class_characteristic_atoms = {}
    
    for class_idx in range(n_classes):
        if class_idx not in class_stats:
            continue
        
        stats = class_stats[class_idx]
        mean_act = stats['mean_activation']
        usage = stats['usage_rate']
        
        # Score: combination of activation strength and usage frequency
        # High score = strong activation AND frequently used in this class
        scores = mean_act * usage
        
        # Get top-k atoms
        top_indices = np.argsort(scores)[-top_k_atoms:][::-1]
        
        # Filter by threshold
        characteristic = []
        for atom_idx in top_indices:
            if scores[atom_idx] >= threshold:
                characteristic.append({
                    'atom_idx': int(atom_idx),
                    'score': float(scores[atom_idx]),
                    'mean_activation': float(mean_act[atom_idx]),
                    'usage_rate': float(usage[atom_idx])
                })
        
        class_characteristic_atoms[class_idx] = characteristic
        
        # Print results
        print(f"\n{class_names[class_idx].upper()} (Class {class_idx}):")
        print(f"  Images analyzed: {stats['n_images']}")
        print(f"  (Using top-{top_channels} influential channels per image)")
        print(f"  Characteristic atoms (score ≥ {threshold}):")
        
        if len(characteristic) > 0:
            for rank, atom_info in enumerate(characteristic, 1):
                print(f"    #{rank:2d}  Atom {atom_info['atom_idx']:3d}  |  "
                      f"Score: {atom_info['score']:6.3f}  |  "
                      f"Avg Activation: {atom_info['mean_activation']:6.3f}  |  "
                      f"Usage: {atom_info['usage_rate']*100:5.1f}%")
        else:
            print(f"    No atoms exceed threshold {threshold}")
    
    # Cross-class analysis: find atoms shared vs unique
    print("\n" + "="*80)
    print("CROSS-CLASS ATOM ANALYSIS")
    print("="*80)
    
    # Get top-5 atoms for each class for overlap analysis
    top_atoms_per_class = {}
    for class_idx, atoms in class_characteristic_atoms.items():
        if len(atoms) > 0:
            top_atoms_per_class[class_idx] = [a['atom_idx'] for a in atoms[:5]]
    
    # Find shared atoms (appear in multiple classes)
    atom_class_map = {}
    for class_idx, atom_list in top_atoms_per_class.items():
        for atom_idx in atom_list:
            if atom_idx not in atom_class_map:
                atom_class_map[atom_idx] = []
            atom_class_map[atom_idx].append(class_idx)
    
    shared_atoms = {atom: classes for atom, classes in atom_class_map.items() 
                    if len(classes) > 1}
    unique_atoms = {atom: classes[0] for atom, classes in atom_class_map.items() 
                    if len(classes) == 1}
    
    print(f"\nShared atoms (used by multiple classes): {len(shared_atoms)}")
    if len(shared_atoms) > 0:
        for atom_idx, class_list in sorted(shared_atoms.items(), 
                                          key=lambda x: len(x[1]), reverse=True)[:10]:
            class_names_list = [class_names[c] for c in class_list]
            print(f"  Atom {atom_idx:3d}: {', '.join(class_names_list)} ({len(class_list)} classes)")
    
    print(f"\nClass-unique atoms (top-5 only in one class): {len(unique_atoms)}")
    for class_idx in range(n_classes):
        unique_for_class = [atom for atom, cls in unique_atoms.items() if cls == class_idx]
        if len(unique_for_class) > 0:
            print(f"  {class_names[class_idx]:12s}: {unique_for_class}")
    
    # Compute inter-class atom overlap
    print("\n" + "="*80)
    print("PAIRWISE CLASS ATOM OVERLAP (Jaccard Similarity)")
    print("="*80)
    
    overlap_matrix = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            if i in top_atoms_per_class and j in top_atoms_per_class:
                atoms_i = set(top_atoms_per_class[i])
                atoms_j = set(top_atoms_per_class[j])
                overlap_matrix[i, j] = jaccard_sim(list(atoms_i), list(atoms_j))
    
    # Print matrix
    print("\n      ", end="")
    for i in range(n_classes):
        print(f"{class_names[i][:4]:>5s}", end="")
    print()
    
    for i in range(n_classes):
        print(f"{class_names[i][:4]:>5s}:", end="")
        for j in range(n_classes):
            if overlap_matrix[i, j] > 0:
                print(f"{overlap_matrix[i, j]:5.2f}", end="")
            else:
                print(f"  -  ", end="")
        print()
    
    # Visualize overlap matrix
    visualize_atom_overlap_matrix(overlap_matrix, class_names)
    
    return {
        'class_stats': class_stats,
        'characteristic_atoms': class_characteristic_atoms,
        'shared_atoms': shared_atoms,
        'unique_atoms': unique_atoms,
        'overlap_matrix': overlap_matrix,
        'class_names': class_names  # Include class names in return value
    }

In [ ]:
atom_analysis = analyze_class_characteristic_atoms(
        results=analysis_result['results'],
        n_atoms=len(D),
        top_k_atoms=10,
        threshold=0.1
    )

In [ ]:
plt.figure(figsize=(2, 2))
plt.imshow(D[107].cpu().numpy(), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
atom_analysis.keys()

In [ ]:
def visualize_unique_atoms_grid(atom_analysis: dict, D: torch.Tensor, 
                                 figsize: tuple = (6, 6), cmap: str = 'gray'):
    """
    Visualize the first unique atom for each class in a 3x3 grid.
    
    Args:
        atom_analysis: Output from analyze_class_characteristic_atoms containing 'unique_atoms'
        D: Dictionary tensor of atoms (n_atoms, H, W)
        figsize: Figure size
        cmap: Colormap for visualization
    """
    unique_atoms = atom_analysis['unique_atoms']
    class_names = atom_analysis['class_names']
    
    # Group unique atoms by class
    class_unique_atoms = {}
    for atom_idx, class_idx in unique_atoms.items():
        if class_idx not in class_unique_atoms:
            class_unique_atoms[class_idx] = []
        class_unique_atoms[class_idx].append(atom_idx)
    
    # Get classes with unique atoms (sorted by class index)
    classes_with_unique = sorted(class_unique_atoms.keys())
    n_classes_with_unique = len(classes_with_unique)
    
    print(f"\nVisualizing {n_classes_with_unique} classes with unique atoms")
    
    # Create 3x3 grid
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    axes = axes.flatten()
    
    for idx, class_idx in enumerate(classes_with_unique):
        # if idx >= 9:  # Only show first 9
        #     break
            
        ax = axes[idx]
        
        # Get first unique atom for this class
        first_atom_idx = class_unique_atoms[class_idx][0]
        atom = D[first_atom_idx].cpu().numpy()
        
        # Display atom
        im = ax.imshow(atom, cmap=cmap)
        ax.set_title(f"{class_names[class_idx]}\nAtom {first_atom_idx}", 
                     fontsize=6, fontweight='bold')
        ax.axis('off')
        
        # Add colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # Hide unused subplots
    for idx in range(n_classes_with_unique, 9):
        axes[idx].axis('off')
    
    plt.suptitle('Class-Unique Atoms (First Unique Atom per Class)', 
                 fontsize=10, fontweight='bold', y=0.98)
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/unique_atoms_grid.png', dpi=150, bbox_inches='tight')
    print("Saved visualization to: figures/unique_atoms_grid.png")
    plt.show()


# Usage example:
visualize_unique_atoms_grid(atom_analysis, D)

In [ ]:
from src.atom import AtomVisualizer


# ============================================================
# Usage Example
# ============================================================

def visualize_multiple_atoms(model, target_layer, D, atom_indices, device='cuda'):
    """Visualize multiple atoms."""
    visualizer = AtomVisualizer(model, target_layer, device)
    
    for atom_idx in atom_indices:
        visualizer.visualize_atom_comparison(
            atom_idx=atom_idx,
            D=D,
            input_size=(3, 224, 224),
            save_path=f'atom_{atom_idx}_visualization.png'
        )



"""

In [ ]:
# Create visualizer
visualizer = AtomVisualizer(model, target_layer, device)

# Ensure D has a channel dimension if required by the visualizer
if D.dim() == 3:
    D_for_vis = D.unsqueeze(1)  # shape: [n_atoms, 1, H, W]
else:
    D_for_vis = D

# Visualize specific atom (e.g., atom 69 that you showed)
visualizer.visualize_atom_comparison(
    atom_idx=69,
    D=D_for_vis,
    input_size=(3, 224, 224),
    # save_path='atom_69_comparison.png'
)

In [ ]:
D[1].shape

In [ ]:
!pip install -q opencv-python

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import torchvision.transforms as T
from typing import Optional, Tuple, List
import os

# ============================================================================
# METHOD 1: SALIENCY MAP / GRADIENT-BASED VISUALIZATION
# ============================================================================

def generate_gradient_visualization(
    model: nn.Module,
    target_layer_idx: int,
    atom_mask: torch.Tensor,
    device: str = 'cuda'
) -> np.ndarray:
    """
    Generate gradient-based visualization showing which input regions 
    are important for activating the atom pattern.
    
    This is a simpler alternative to Guided Backprop that avoids hook issues.
    
    Args:
        model: Neural network model
        target_layer_idx: Index of target layer in feature_extractor
        atom_mask: Atom pattern (h, w) to focus on
        device: 'cuda' or 'cpu'
        
    Returns:
        Gradient visualization (H, W, 3)
    """
    model.eval()
    
    # Create random input image
    input_img = torch.randn(1, 3, 224, 224, device=device) * 0.1
    input_img.requires_grad_(True)
    
    # Storage for activation
    activation = None
    
    def hook_fn(module, input, output):
        nonlocal activation
        activation = output
    
    # Register forward hook
    target_layer = model.feature_extractor[target_layer_idx]
    handle = target_layer.register_forward_hook(hook_fn)
    
    try:
        # Forward pass
        _ = model(input_img)
        
        if activation is None:
            return np.zeros((224, 224, 3))
        
        # Prepare atom tensor
        atom_tensor = atom_mask.to(device).unsqueeze(0).unsqueeze(0)
        
        # Compute spatial activation
        spatial_act = torch.mean(activation, dim=1, keepdim=True)  # (1, 1, h, w)
        
        # Resize atom if needed
        if spatial_act.shape[2:] != atom_tensor.shape[2:]:
            atom_tensor = F.interpolate(
                atom_tensor, 
                size=spatial_act.shape[2:], 
                mode='bilinear', 
                align_corners=False
            )
        
        # Loss: maximize atom-weighted activation
        loss = (spatial_act * atom_tensor).sum()
        
        # Backward to get gradients
        loss.backward()
        
        if input_img.grad is None:
            return np.zeros((224, 224, 3))
        
        # Get gradients
        gradients = input_img.grad.data.cpu().numpy()[0]  # (3, H, W)
        
        # Take absolute value and average across channels for saliency
        saliency = np.abs(gradients).max(axis=0)  # (H, W)
        
        # Normalize
        saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
        
        # Convert to RGB by repeating
        saliency_rgb = np.stack([saliency] * 3, axis=-1)
        
        return saliency_rgb
        
    except Exception as e:
        print(f"Error in gradient visualization: {e}")
        return np.zeros((224, 224, 3))
    finally:
        handle.remove()


# ============================================================================
# METHOD 2: ACTIVATION MAXIMIZATION (From your notebook)
# ============================================================================

def denormalize_tensor(tensor):
    """Convert normalized tensor back to RGB image for display."""
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)
    tensor = tensor * std + mean
    img_np = torch.clamp(tensor, 0, 1).permute(1, 2, 0).cpu().numpy()
    return img_np

def visualize_atom_activation_maximization(
    model, 
    atom, 
    layer_index, 
    steps=300, 
    lr=0.1, 
    device='cuda',
    verbose=False
):
    """
    Generate input image that maximally activates a specific atom.
    (Improved version from your notebook)
    
    Args:
        model: Neural network model
        atom: Atom pattern (h, w)
        layer_index: Index of target layer in feature_extractor
        steps: Number of optimization steps
        lr: Learning rate
        device: 'cuda' or 'cpu'
        verbose: Print progress
        
    Returns:
        Optimized image (H, W, 3) in [0, 1] range
    """
    model.eval()
    
    # Initialize random input with small noise
    random_img = torch.randn(1, 3, 224, 224, device=device) * 0.01
    random_img.requires_grad_(True)
    
    optimizer = optim.Adam([random_img], lr=lr)
    atom_tensor = torch.tensor(atom, device=device).float()
    
    target_activations = []
    def hook_fn(module, input, output):
        target_activations.append(output)
    
    handle = model.feature_extractor[layer_index].register_forward_hook(hook_fn)
    
    try:
        for i in range(steps):
            optimizer.zero_grad()
            target_activations.clear()
            
            # Forward pass
            _ = model(random_img)
            act = target_activations[0].squeeze(0)  # (C, h, w)
            
            # Compute spatial activation map
            spatial_activation = torch.mean(act, dim=0)  # (h, w)
            
            # Loss: maximize similarity with atom pattern
            # Element-wise product favors regions where both are high
            loss = -torch.sum(spatial_activation * atom_tensor)
            
            # Add regularization
            l2_reg = 1e-4 * torch.norm(random_img)
            tv_reg = 1e-3 * total_variation(random_img)
            
            total_loss = loss + l2_reg + tv_reg
            total_loss.backward()
            optimizer.step()
            
            # Clamp to reasonable range
            with torch.no_grad():
                random_img.data = torch.clamp(random_img.data, -2.5, 2.5)
            
            # Apply blur every 4 iterations to reduce noise
            if (i + 1) % 4 == 0:
                with torch.no_grad():
                    random_img.data = gaussian_blur(random_img.data)
            
            if verbose and (i + 1) % 50 == 0:
                print(f"Step {i+1}/{steps}, Loss: {total_loss.item():.4f}")
                
    finally:
        handle.remove()
    
    return denormalize_tensor(random_img.squeeze(0).detach())

def total_variation(img):
    """Total variation loss for smoothness."""
    tv_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).sum()
    tv_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).sum()
    return tv_h + tv_w

def gaussian_blur(img, kernel_size=3):
    """Simple Gaussian blur using average pooling."""
    return F.avg_pool2d(img, kernel_size, stride=1, padding=kernel_size // 2)


# ============================================================================
# METHOD 3: ACTIVATION OVERLAY (From your notebook)
# ============================================================================

def create_atom_overlay_on_image(model, atom, image_path, layer_idx, device):
    """
    Create overlay showing where atom activates on a real image.
    
    Args:
        model: Neural network
        atom: Atom pattern (h, w)
        image_path: Path to input image
        layer_idx: Target layer index
        device: 'cuda' or 'cpu'
        
    Returns:
        Overlay image (H, W, 3) as numpy array
    """
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    try:
        # Load and preprocess image
        img_pil = Image.open(image_path).convert('RGB')
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        
        # Get activation at target layer
        target_activations = []
        def hook_fn(module, input, output):
            target_activations.append(output)
        
        handle = model.feature_extractor[layer_idx].register_forward_hook(hook_fn)
        
        with torch.no_grad():
            _ = model(img_tensor)
        
        handle.remove()
        
        # Get activation map
        act = target_activations[0].squeeze(0)  # (C, h, w)
        spatial_act = torch.mean(act, dim=0)  # (h, w)
        
        # Apply atom mask
        atom_tensor = torch.tensor(atom, device=device) if not torch.is_tensor(atom) else atom
        atom_act_map = spatial_act * atom_tensor
        atom_act_map = torch.relu(atom_act_map)
        
        # Normalize
        if atom_act_map.max() > 0:
            atom_act_map /= atom_act_map.max()
        
        # Resize to 224x224 and create heatmap
        heatmap = atom_act_map.cpu().numpy()
        heatmap = cv2.resize(heatmap, (224, 224))
        heatmap = np.uint8(255 * heatmap)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        
        # Prepare original image
        img_orig = np.array(img_pil.resize((224, 224)))
        img_orig = cv2.cvtColor(img_orig, cv2.COLOR_RGB2BGR)
        
        # Blend
        overlay = cv2.addWeighted(img_orig, 0.6, heatmap, 0.4, 0)
        overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
        
        return overlay
        
    except Exception as e:
        print(f"Error creating overlay: {e}")
        return np.zeros((224, 224, 3), dtype=np.uint8)


# ============================================================================
# COMPREHENSIVE VISUALIZATION: All 4 Methods Side-by-Side
# ============================================================================

def visualize_atom_comprehensive(
    model,
    atom_idx: int,
    D: torch.Tensor,
    layer_idx: int,
    device: str,
    sample_image_path: Optional[str] = None,
    class_name: str = "Unknown",
    save_path: str = None
):
    """
    Comprehensive atom visualization with 4 panels:
    1. Atom pattern (abstract heatmap)
    2. Guided Backpropagation
    3. Activation Maximization
    4. Real Image Overlay (if sample provided)
    
    Args:
        model: Neural network
        atom_idx: Index of atom in dictionary D
        D: Dictionary tensor of atoms
        layer_idx: Target layer index
        device: 'cuda' or 'cpu'
        sample_image_path: Optional path to sample image
        class_name: Name of class for title
        save_path: Where to save the figure
    """
    print(f"\n{'='*70}")
    print(f"Visualizing Atom {atom_idx} for class '{class_name}'")
    print(f"{'='*70}")
    
    # Get atom data
    if torch.is_tensor(D):
        atom_data = D[atom_idx].cpu().numpy()
        atom_tensor = D[atom_idx]
    else:
        atom_data = D[atom_idx]
        atom_tensor = torch.tensor(atom_data)
    
    # Create figure with 4 panels
    n_panels = 4 if sample_image_path else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(5*n_panels, 5))
    
    # Panel 1: Atom Heatmap
    print("Panel 1: Rendering atom pattern...")
    im = axes[0].imshow(atom_data, cmap='hot')
    axes[0].set_title(f'Atom {atom_idx}\n(Abstract Pattern)', fontweight='bold')
    axes[0].axis('off')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Panel 2: Guided Backpropagation
    print("Panel 2: Running Guided Backpropagation...")
    try:
        # Create a sample input for backprop
        sample_input = torch.randn(1, 3, 224, 224, device=device) * 0.1
        guided_bp = GuidedBackprop(model, layer_idx)
        backprop_result = guided_bp.generate(sample_input, atom_tensor)
        
        axes[1].imshow(backprop_result)
        axes[1].set_title('Guided Backpropagation\n(Gradient Tracing)', fontweight='bold')
        axes[1].axis('off')
    except Exception as e:
        print(f"Guided backprop failed: {e}")
        axes[1].text(0.5, 0.5, 'Method Failed', ha='center', va='center')
        axes[1].axis('off')
    
    # Panel 3: Activation Maximization
    print("Panel 3: Running Activation Maximization (this may take ~30s)...")
    opt_result = visualize_atom_activation_maximization(
        model, atom_data, layer_idx, steps=300, lr=0.1, device=device, verbose=False
    )
    axes[2].imshow(opt_result)
    axes[2].set_title('Activation Maximization\n(Optimized Input)', fontweight='bold')
    axes[2].axis('off')
    
    # Panel 4: Real Image Overlay (if available)
    if sample_image_path:
        print("Panel 4: Creating overlay on real image...")
        overlay = create_atom_overlay_on_image(model, atom_data, sample_image_path, layer_idx, device)
        axes[3].imshow(overlay)
        axes[3].set_title(f'Activation on Sample\n({os.path.basename(sample_image_path)})', 
                         fontweight='bold')
        axes[3].axis('off')
    
    plt.suptitle(f'Class: {class_name} | Atom {atom_idx} Visualization', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ Saved to: {save_path}")
    
    plt.show()
    print("Done!\n")


# ============================================================================
# BATCH VISUALIZATION: For Multiple Classes
# ============================================================================

def visualize_class_unique_atoms_comprehensive(
    atom_analysis: dict,
    D: torch.Tensor,
    model,
    layer_idx: int,
    device: str,
    subset_paths: List[Tuple[str, int]],
    output_dir: str = 'figures'
):
    """
    Create comprehensive visualizations for all class-unique atoms.
    
    Args:
        atom_analysis: Output from analyze_class_characteristic_atoms
        D: Dictionary of atoms
        model: Neural network
        layer_idx: Target layer index
        device: 'cuda' or 'cpu'
        subset_paths: List of (image_path, label) tuples
        output_dir: Directory to save outputs
    """
    unique_atoms = atom_analysis['unique_atoms']
    class_names = atom_analysis['class_names']
    
    # Group unique atoms by class
    class_unique_atoms = {}
    for atom_idx, class_idx in unique_atoms.items():
        if class_idx not in class_unique_atoms:
            class_unique_atoms[class_idx] = []
        class_unique_atoms[class_idx].append(atom_idx)
    
    classes_with_unique = sorted(class_unique_atoms.keys())
    
    print(f"\n{'='*70}")
    print(f"Creating comprehensive visualizations for {len(classes_with_unique)} classes")
    print(f"{'='*70}\n")
    
    for class_idx in classes_with_unique:
        # Get first unique atom for this class
        atom_idx = class_unique_atoms[class_idx][0]
        
        # Find sample image for this class
        sample_img_path = None
        for path, label in subset_paths:
            if label == class_idx:
                sample_img_path = path
                break
        
        # Create visualization
        save_path = os.path.join(output_dir, f'atom_{atom_idx}_class_{class_names[class_idx]}.png')
        
        visualize_atom_comprehensive(
            model=model,
            atom_idx=atom_idx,
            D=D,
            layer_idx=layer_idx,
            device=device,
            sample_image_path=sample_img_path,
            class_name=class_names[class_idx],
            save_path=save_path
        )


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

"""
# In your notebook:

# 1. Visualize a single atom comprehensively
visualize_atom_comprehensive(
    model=model,
    atom_idx=107,  # Example: atom 107
    D=D,
    layer_idx=5,  # target_layer index
    device=device,
    sample_image_path="data/subset/church/1.JPEG",
    class_name="church",
    save_path="figures/atom_107_comprehensive.png"
)

# 2. Visualize all unique atoms for all classes
visualize_class_unique_atoms_comprehensive(
    atom_analysis=atom_analysis,
    D=D,
    model=model,
    layer_idx=5,
    device=device,
    subset_paths=subset_paths,
    output_dir='figures/comprehensive'
)
"""

In [ ]:
visualize_atom_comprehensive(
    model=model,
    atom_idx=107,  # Example: atom 107
    D=D,
    layer_idx=5,  # target_layer index
    device=device,
    sample_image_path="data/subset/church/1.jpeg",
    class_name="church",
    save_path="figures/atom_107_comprehensive.png"
)